GPT-2 Model Configuration

In [3]:
GPT_CONFIG_124M = {
    "vocab_size": 50527,
    "context_length": 256,
    "stride": 128,
    "embedding_dim": 768,
    "num_layers": 12,
    "num_heads": 12,
    "dropout": 0.1,
    "qkv_bias": False
}

Load Dataset

In [6]:
file_path = "../romeo-juliet.txt"

with open(file_path, 'r', encoding = 'utf-8') as file:
    text_data = file.read()

Tokenization

In [7]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [9]:
token_ids = tokenizer.encode(text_data)

In [10]:
len(token_ids)

50325

Implementing Dataloader

In [13]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_len, stride):

        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text)

        for i in range(0, len(token_ids) - max_len, stride):
            input_chunk = token_ids[i:i+max_len]
            target_chunk = token_ids[i+1:i+max_len+1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]
    
def create_dataloader(text, tokenizer, batch_size, context_len, stride, shuffle, drop_last, num_workers):

    dataset = GPTDataset(text, tokenizer, context_len, stride)

    dataloader = DataLoader(

        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers    
    )

    return dataloader

In [14]:
train_ratio = 0.8

train_data = text_data[:int(train_ratio*len(text_data))]
test_data = text_data[int(train_ratio*len(text_data)):]

train_dataloader = create_dataloader(

    text=train_data,
    tokenizer=tokenizer,
    batch_size=128,
    context_len=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["stride"],
    shuffle=True,
    drop_last=True,
    num_workers=0

)

test_dataloader = create_dataloader(
    
    text=test_data,
    tokenizer=tokenizer,
    batch_size=64,
    context_len=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["stride"],
    shuffle=False,
    drop_last=False,
    num_workers=0

)

In [15]:
for x, y in train_dataloader:
    print(x.shape, y.shape)

for x, y in test_dataloader:
    print(x.shape, y.shape)


torch.Size([128, 256]) torch.Size([128, 256])
torch.Size([128, 256]) torch.Size([128, 256])
torch.Size([64, 256]) torch.Size([64, 256])
torch.Size([2, 256]) torch.Size([2, 256])


Multi-Head Attention

In [17]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, num_heads, context_len, dropout, qkv_bias):
        super().__init__()

        assert(d_out % num_heads ==0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = (d_out // num_heads)

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(d_out, d_out)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_score = queries @ keys.transpose(2, 3)
        
        masked_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_score.masked_fill_(masked_bool, -torch.inf)

        attn_weights = torch.softmax(attn_score / (keys.shape[-1] ** 0.5), dim=1)
        attn_weights = self.dropout(attn_weights)

        context_vector = attn_weights @ values
        context_vector = context_vector.transpose(1, 2)

        context_vector = context_vector.contiguous().view(b, num_tokens, self.d_out)
        context_vector = self.out_proj(context_vector)

        return context_vector
        


Layer Normalization, GELU Activation Function and Feed Forward Neural Network

In [22]:
class LayerNormaliztion(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()

        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)
        x_norm = (x - mean)/torch.sqrt(var + self.eps)

        return self.scale * x_norm + self.shift


class GELUActivation(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))
    
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["embedding_dim"], 4*cfg["embedding_dim"]),
            GELUActivation(),
            nn.Linear(4*cfg["embedding_dim"], cfg["embedding_dim"])

        )
        
    def forward(self, x):
        return self.layers(x)

Transformer Block

In [25]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            d_in=cfg["embedding_dim"],
            d_out=cfg["embedding_dim"],
            num_heads=cfg["num_heads"],
            context_len=cfg["context_length"],
            dropout=cfg["dropout"],
            qkv_bias=cfg["qkv_bias"]
        )

        self.norm1 = LayerNormaliztion(cfg["embedding_dim"])
        self.norm2 = LayerNormaliztion(cfg["embedding_dim"])
        self.ff = FeedForward(cfg)
        self.dropout = nn.Dropout(cfg["dropout"])
    
    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.dropout(x)
        x = x + shortcut

        return x

GPT Model Architecture

In [26]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["embedding_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["embedding_dim"])
        self.dropout = nn.Dropout(cfg["dropout"])

        self.transformers = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["num_layers"])]
        )

        self.final_norm = LayerNormaliztion(cfg["embedding_dim"])
        self.final_output = nn.Linear(cfg["embedding_dim"], cfg["vocab_size"], bias=False)

    def forward(self, x):
        batch_size, num_tokens, vocab_size = x.shape

        tok_emb = self.tok_emb(x)
        pos_emb = self.pos_emb(x)
        x = tok_emb + pos_emb
        x = self.dropout(x)

        x = self.transformers(x)

        x = self.final_norm(x)
        logits = self.final_output(x)

        return logits

In [27]:
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50527, 768)
  (pos_emb): Embedding(256, 768)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformers): Sequential(
    (0): TransformerBlock(
      (attention): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (norm1): LayerNormaliztion()
      (norm2): LayerNormaliztion()
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELUActivation()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attention): MultiHeadAttention(
    

Loss Function